# E06 — a mesma estatística, lida ao contrário

O capítulo 4 fechou com uma pergunta: a estatística que responde *que conjunto ainda vale* e a
que responde *em que instante ele deixou de valer* são a mesma, lida ao contrário?

**O que se testa.** Para cada dia, três sinais olhando para trás contra os rompimentos dos
\`bloco\` dias **seguintes**:

1. o **movimento do corte** — a estimativa se mexeu nos últimos dias;
2. a **contagem recente** — quantos rompimentos houve (o vigia do capítulo 2);
3. o **nível do corte** — a própria estimativa, e a sua versão relativa à oscilação de agora.

**Os controles.** Um mundo que nunca muda, onde o futuro tem de ser plano em todos os sinais, e
uma segunda série, para o resultado não ser um acidente de um mercado.

**A separação.** O nível do corte é parente próximo da oscilação do último ano, e *volatilidade
prevê volatilidade* é coisa antiga. O caderno mede o nível **dentro** de faixas de oscilação,
que é o único jeito de saber se ele diz algo além disso.

**Convenções** (AGENTS.md §7 e §9): parâmetros no topo marcados "brinque com", algoritmo em
frevolab, resultado em lab/resultados/E06_estabilidade.json.

In [1]:
# <- brinque com: SERIE, SEGUNDA, JANELA, POSTO, BLOCO, FAIXAS, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, estabilidade, graficos, promessa, volatilidade

RAIZ = Path.cwd()
SERIE = "sp500.csv"
SEGUNDA = "ibov.csv"
JANELA = 252          # o corte do capitulo 1
POSTO = 13            # o decimo terceiro pior, como no capitulo 1
BLOCO = 60            # o bloco do capitulo 2
FAIXAS = 3            # em quantas faixas de oscilacao o nivel do corte e testado
SEMENTE = 71

retornos = volatilidade.retornos_log(dados.carregar_serie(SERIE))
print("frevolab %s | %s: %d dias, de %s a %s" % (
    frevolab.VERSAO, SERIE, len(retornos), retornos.index.min().date(), retornos.index.max().date()))

frevolab 0.1.0 | sp500.csv: 6718 dias, de 2000-01-04 a 2026-09-21


## Os três sinais contra o que vem

In [2]:
# Os tres sinais, o alvo e a oscilacao de agora: tudo olhando para tras, nada para frente.
corte = promessa.corte(retornos, JANELA, POSTO / JANELA).to_numpy()
x = retornos.to_numpy()
rompe = (x < corte).astype(float)
rompe[np.isnan(corte)] = np.nan

futuro = pd.Series(rompe).rolling(BLOCO).sum().shift(-BLOCO).to_numpy()
contagem = pd.Series(rompe).rolling(BLOCO).sum().to_numpy()
movimento = np.full(len(x), np.nan)
movimento[BLOCO:] = (corte[BLOCO:] - corte[:-BLOCO]) / np.abs(corte[:-BLOCO])
oscilacao = pd.Series(x).rolling(BLOCO).std().to_numpy()
relativo = corte / oscilacao

util = ~np.isnan(futuro) & ~np.isnan(movimento) & ~np.isnan(contagem) & ~np.isnan(relativo)
sinais = {"movimento do corte": movimento, "contagem recente": contagem,
          "nivel do corte": corte, "nivel sobre a oscilacao": relativo}
fut = futuro[util]
print("dias utilizaveis: %d | futuro medio %.2f | pior futuro %d" % (util.sum(), fut.mean(), fut.max()))


posto = estabilidade.posto

decimos = lambda sinal: estabilidade.perfil_por_decimo(sinal[util], fut)


print()
for nome, sinal in sinais.items():
    perfil = decimos(sinal)
    print("%-24s posto %+.3f | 1o decimo %.2f | 10o decimo %.2f | %s"
          % (nome, posto(sinal[util], fut), perfil[0], perfil[-1],
             " ".join("%.2f" % v for v in perfil)))

dias utilizaveis: 6346 | futuro medio 3.07 | pior futuro 20

movimento do corte       posto +0.040 | 1o decimo 3.29 | 10o decimo 3.05 | 3.29 2.78 3.22 2.67 2.35 2.86 3.43 3.42 3.64 3.05
contagem recente         posto +0.212 | 1o decimo 1.65 | 10o decimo 3.85 | 1.65 2.02 2.88 2.50 3.31 4.52 3.00 3.78 3.19 3.85
nivel do corte           posto +0.403 | 1o decimo 1.00 | 10o decimo 4.68 | 1.00 1.84 2.34 3.33 2.45 3.85 3.69 3.79 3.74 4.68
nivel sobre a oscilacao  posto +0.263 | 1o decimo 1.44 | 10o decimo 3.98 | 1.44 2.18 2.55 2.94 2.44 3.91 3.62 3.82 3.82 3.98


## O nível do corte diz algo além da oscilação de agora?

In [3]:
# O nivel do corte diz algo alem da oscilacao de agora? Teste dentro de faixas dela.
faixas = np.array_split(np.argsort(oscilacao[util]), FAIXAS)
print("posto do nivel do corte com a oscilacao de agora: %+.3f" % posto(corte[util], oscilacao[util]))
print()
print("%-14s %-8s %-22s %-22s" % ("faixa de oscilacao", "dias", "nivel do corte", "contagem recente"))
for i, faixa in enumerate(faixas):
    selecao = np.zeros(util.sum(), dtype=bool)
    selecao[faixa] = True
    print("%-14s %-8d %-22s %-22s" % (
        "faixa %d" % (i + 1), selecao.sum(),
        "posto %+.3f" % posto(corte[util][selecao], fut[selecao]),
        "posto %+.3f" % posto(contagem[util][selecao], fut[selecao])))

posto do nivel do corte com a oscilacao de agora: -0.763

faixa de oscilacao dias     nivel do corte         contagem recente      
faixa 1        2116     posto +0.275           posto +0.037          
faixa 2        2115     posto +0.415           posto +0.425          
faixa 3        2115     posto +0.499           posto +0.371          


## O controle: um mundo que nunca muda

In [4]:
# O controle: um mundo que nunca muda, com a mesma rotina. Todo sinal tem de ficar plano.
sorteio = np.random.default_rng(SEMENTE)
calmo = pd.Series(sorteio.normal(0.0, 0.01, len(retornos)), index=retornos.index)
xc = calmo.to_numpy()
corte_c = promessa.corte(calmo, JANELA, POSTO / JANELA).to_numpy()
rompe_c = (xc < corte_c).astype(float)
rompe_c[np.isnan(corte_c)] = np.nan
futuro_c = pd.Series(rompe_c).rolling(BLOCO).sum().shift(-BLOCO).to_numpy()
contagem_c = pd.Series(rompe_c).rolling(BLOCO).sum().to_numpy()
mov_c = np.full(len(xc), np.nan)
mov_c[BLOCO:] = (corte_c[BLOCO:] - corte_c[:-BLOCO]) / np.abs(corte_c[:-BLOCO])
util_c = ~np.isnan(futuro_c) & ~np.isnan(mov_c) & ~np.isnan(contagem_c)
print("mundo que nunca muda: futuro medio %.2f" % futuro_c[util_c].mean())
for nome, sinal in (("movimento", mov_c), ("contagem", contagem_c), ("nivel", corte_c)):
    print("   %-12s posto %+.3f" % (nome, posto(sinal[util_c], futuro_c[util_c])))

mundo que nunca muda: futuro medio 3.10
   movimento    posto +0.072
   contagem     posto +0.038
   nivel        posto +0.414


## A segunda série

In [5]:
# A segunda serie: se o resultado so vale num mercado, ele nao vale.
outros = volatilidade.retornos_log(dados.carregar_serie(SEGUNDA))
xo = outros.to_numpy()
corte_o = promessa.corte(outros, JANELA, POSTO / JANELA).to_numpy()
rompe_o = (xo < corte_o).astype(float)
rompe_o[np.isnan(corte_o)] = np.nan
futuro_o = pd.Series(rompe_o).rolling(BLOCO).sum().shift(-BLOCO).to_numpy()
contagem_o = pd.Series(rompe_o).rolling(BLOCO).sum().to_numpy()
mov_o = np.full(len(xo), np.nan)
mov_o[BLOCO:] = (corte_o[BLOCO:] - corte_o[:-BLOCO]) / np.abs(corte_o[:-BLOCO])
util_o = ~np.isnan(futuro_o) & ~np.isnan(mov_o) & ~np.isnan(contagem_o) & ~np.isnan(corte_o)
print("%s: %d dias | futuro medio %.2f" % (SEGUNDA, util_o.sum(), futuro_o[util_o].mean()))
for nome, sinal in (("movimento", mov_o), ("contagem", contagem_o), ("nivel", corte_o)):
    print("   %-12s posto %+.3f" % (nome, posto(sinal[util_o], futuro_o[util_o])))

ibov.csv: 6248 dias | futuro medio 3.04
   movimento    posto +0.066
   contagem     posto +0.123
   nivel        posto +0.347


## As figuras

In [6]:
# Figura 1: o futuro medio em cada decimo dos quatro sinais.
fig, eixo = plt.subplots(figsize=(9.4, 4.4))
largura = 0.2
posicoes = np.arange(10)
cores = ("#b03a2e", "#1f4e79", "#b8860b", "#4a7c59")
for i, (nome, sinal) in enumerate(sinais.items()):
    eixo.bar(posicoes + (i - 1.5) * largura, decimos(sinal), largura, color=cores[i], label=nome)
eixo.axhline(fut.mean(), color="#7f7f7f", ls="--", lw=1.2, label="o futuro medio: %.2f" % fut.mean())
eixo.set_xticks(posicoes)
eixo.set_xticklabels(["%d" % (i + 1) for i in posicoes])
eixo.set_xlabel("decimo do sinal, do menor para o maior")
eixo.set_ylabel("rompimentos nos %d dias seguintes" % BLOCO)
eixo.legend(frameon=False, fontsize=8)
eixo.grid(alpha=0.25, axis="y")
fig.tight_layout()
graficos.salvar(fig, "E06_estabilidade", 1)
plt.close(fig)
print("primeiro e ultimo decimo por sinal: %s" % {n: (round(decimos(s)[0], 2), round(decimos(s)[-1], 2)) for n, s in sinais.items()})

primeiro e ultimo decimo por sinal: {'movimento do corte': (3.29, 3.05), 'contagem recente': (1.65, 3.85), 'nivel do corte': (1.0, 4.68), 'nivel sobre a oscilacao': (1.44, 3.98)}


In [7]:
# Figura 2: o nivel do corte dentro de cada faixa de oscilacao.
fig, eixo = plt.subplots(figsize=(8.4, 4.2))
posicoes = np.arange(FAIXAS)
largura = 0.38
nivel_faixa, contagem_faixa = [], []
for faixa in faixas:
    selecao = np.zeros(util.sum(), dtype=bool)
    selecao[faixa] = True
    nivel_faixa.append(posto(corte[util][selecao], fut[selecao]))
    contagem_faixa.append(posto(contagem[util][selecao], fut[selecao]))
eixo.bar(posicoes - largura / 2, nivel_faixa, largura, color="#b8860b", label="nivel do corte")
eixo.bar(posicoes + largura / 2, contagem_faixa, largura, color="#1f4e79", label="contagem recente")
eixo.axhline(0, color="#7f7f7f", lw=1.0)
eixo.set_xticks(posicoes)
eixo.set_xticklabels(["oscilacao baixa", "media", "alta"][:FAIXAS])
eixo.set_ylabel("correlacao de posto com o futuro")
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25, axis="y")
fig.tight_layout()
graficos.salvar(fig, "E06_estabilidade", 2)
plt.close(fig)
print("posto por faixa: nivel %s | contagem %s"
      % ([round(v, 2) for v in nivel_faixa], [round(v, 2) for v in contagem_faixa]))

posto por faixa: nivel [0.27, 0.42, 0.5] | contagem [0.04, 0.43, 0.37]


## Leitura visual das figuras

Feita nesta sessão abrindo os .png com a ponte de visão (AGENTS.md §9). Observação, não número.

**Figura 1.** Dez grupos de quatro barras, com a linha tracejada do futuro médio atravessando o
painel inteiro. Três das quatro séries sobem da esquerda para a direita; a do movimento do corte
atravessa o gráfico de lado, e os seus dois extremos ficam à mesma altura da linha. O que o eixo
engana: como as barras estão agrupadas por décimo e as alturas são parecidas entre séries, o olho
compara vizinhas e perde a tendência — a leitura certa é seguir uma cor de ponta a ponta, e é isso
que separa a que sobe da que não sobe.

**Figura 2.** Duas barras em cada uma de três faixas de oscilação, e o eixo vertical curto (zero a
meio). Na faixa calma o nível do corte domina e a contagem recente quase não aparece; na média as
duas se igualam; na agitada o nível volta a dominar. O que o eixo engana: com escala pequena,
diferença de quatro décimos parece abismo, e a barra quase ausente da contagem na faixa calma lê-se
como defeito — quando o que ela diz é que a contagem é cega justamente onde nada acontece.


## O que o controle fez com a leitura

O nível do corte parece o melhor dos três sinais: correlação de posto de +0,403 com os rompimentos
dos sessenta dias seguintes, contra +0,212 da contagem e +0,040 do movimento. E sobrevive à
normalização (+0,263 sobre a oscilação de agora) e ao teste dentro das faixas (+0,27, +0,42 e
+0,50 nas três).

O controle diz outra coisa. **Num mundo que nunca muda — sorteados independentes, nada a
prever — o nível do corte dá +0,414.** Mais do que no mercado real. Não há nada para anunciar
naquele mundo, e ainda assim o número aparece.

O motivo é aritmético e não do mundo: o corte é o décimo terceiro pior dos últimos 252 dias, e
os rompimentos dos 60 dias seguintes são contados com cortes que ainda carregam boa parte
daquela mesma janela. As duas pontas se sobrepõem. Um corte fundo quer dizer que o ano teve
dias enormes, o que empurra a linha para longe e **reduz** os rompimentos que vêm — e é isso, e
não uma descoberta sobre o mercado, que a correlação mede.

Sobram dois números que o controle não explica: a contagem recente dá +0,212 no mercado contra
+0,038 no mundo parado, e a segunda série repete a ordem (nível +0,347, contagem +0,123,
movimento +0,066). O movimento do corte, que é o sinal que a prática vigia, não vale nada em
nenhum dos dois mundos.

A lição do caderno está no controle, e não no resultado: **sem o mundo que nunca muda, a leitura
do nível do corte entraria no capítulo como descoberta.**


In [8]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
perfis = {nome: decimos(sinal) for nome, sinal in sinais.items()}
resultado = {
    "estabilidade_dias": int(util.sum()),
    "estabilidade_futuro_medio": float(fut.mean()),
    "estabilidade_futuro_pior": int(fut.max()),
    "estabilidade_movimento_posto": posto(movimento[util], fut),
    "estabilidade_contagem_posto": posto(contagem[util], fut),
    "estabilidade_nivel_posto": posto(corte[util], fut),
    "estabilidade_relativo_posto": posto(relativo[util], fut),
    "estabilidade_movimento_primeiro_decimo": perfis["movimento do corte"][0],
    "estabilidade_movimento_ultimo_decimo": perfis["movimento do corte"][-1],
    "estabilidade_contagem_primeiro_decimo": perfis["contagem recente"][0],
    "estabilidade_contagem_ultimo_decimo": perfis["contagem recente"][-1],
    "estabilidade_nivel_primeiro_decimo": perfis["nivel do corte"][0],
    "estabilidade_nivel_ultimo_decimo": perfis["nivel do corte"][-1],
    "estabilidade_relativo_primeiro_decimo": perfis["nivel sobre a oscilacao"][0],
    "estabilidade_relativo_ultimo_decimo": perfis["nivel sobre a oscilacao"][-1],
    "estabilidade_nivel_com_oscilacao_posto": posto(corte[util], oscilacao[util]),
    "estabilidade_controle_futuro_medio": float(futuro_c[util_c].mean()),
    "estabilidade_controle_movimento_posto": posto(mov_c[util_c], futuro_c[util_c]),
    "estabilidade_controle_contagem_posto": posto(contagem_c[util_c], futuro_c[util_c]),
    "estabilidade_controle_nivel_posto": posto(corte_c[util_c], futuro_c[util_c]),
    "estabilidade_segunda_posto": posto(corte_o[util_o], futuro_o[util_o]),
    "estabilidade_segunda_contagem_posto": posto(contagem_o[util_o], futuro_o[util_o]),
    "estabilidade_segunda_movimento_posto": posto(mov_o[util_o], futuro_o[util_o]),
}
for i, (faixa, valor) in enumerate(zip(faixas, nivel_faixa), start=1):
    extenso = {1: "baixa", 2: "media", 3: "alta"}[i]
    resultado["estabilidade_faixa_%s_nivel_posto" % extenso] = float(valor)
    resultado["estabilidade_faixa_%s_contagem_posto" % extenso] = float(contagem_faixa[i - 1])

caminho = Path("lab/resultados/E06_estabilidade.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E06_estabilidade.json gravado | 29 grandezas
